# ASIC Dynamic First Glance

Load the raw ASIC dynamic files, compare schemas across hospitals, apply the shared translation map, inspect key value checks, and review a first harmonized dynamic view.

In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text


def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "icu_data_platform").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing src/icu_data_platform")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from icu_data_platform.sources.asic.qc.dynamic_checks import (
    DEFAULT_DYNAMIC_DATA_DIR,
    DEFAULT_TRANSLATION_PATH,
    build_canonical_to_raw,
    build_harmonized_dynamic_table,
    find_non_numeric_value_issues,
    flag_cross_hospital_distribution_issues,
    load_dynamic_tables,
    load_dynamic_translation,
    normalize_missing,
    numeric_distribution_summary,
)

RAW_DYNAMIC_DIR = DEFAULT_DYNAMIC_DATA_DIR
TRANSLATION_PATH = DEFAULT_TRANSLATION_PATH
RAW_DYNAMIC_DIR, TRANSLATION_PATH


(PosixPath('/Users/joanameyer/data/asic/raw_sample10'),
 PosixPath('/Users/joanameyer/repository/icu-data-platform/src/icu_data_platform/sources/asic/column_translation.json'))

In [2]:
def empty_columns(df):
    return [column for column in df.columns if normalize_missing(df[column]).isna().all()]


def example_values(series, max_values=8):
    normalized = normalize_missing(series)
    unique_values = normalized.dropna().astype(str).unique().tolist()
    preview = unique_values[:max_values]
    if len(unique_values) > max_values:
        preview.append("...")
    return preview


def value_preview(df, max_values=8):
    rows = []
    for column in df.columns:
        normalized = normalize_missing(df[column])
        non_null = normalized.dropna()
        rows.append(
            {
                "column": column,
                "dtype": str(df[column].dtype),
                "non_null_count": int(non_null.shape[0]),
                "n_unique_non_null": int(non_null.nunique()),
                "example_values": example_values(df[column], max_values=max_values),
            }
        )
    return pd.DataFrame(rows)


def numeric_summary(series):
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return {
            "n": 0,
            "min": pd.NA,
            "q1": pd.NA,
            "median": pd.NA,
            "q3": pd.NA,
            "max": pd.NA,
            "iqr": pd.NA,
            "example_values": [],
        }
    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)
    return {
        "n": int(numeric.shape[0]),
        "min": float(numeric.min()),
        "q1": float(q1),
        "median": float(numeric.median()),
        "q3": float(q3),
        "max": float(numeric.max()),
        "iqr": float(q3 - q1),
        "example_values": numeric.unique()[:10].tolist(),
    }


## Raw Dynamic Files

Load all hospital-level ASIC dynamic CSV files and inspect the raw dataset shape.

In [3]:
dynamic_tables = load_dynamic_tables(RAW_DYNAMIC_DIR)
dynamic_translation = load_dynamic_translation(TRANSLATION_PATH)
canonical_to_raw = build_canonical_to_raw(dynamic_translation)

dataset_overview = pd.DataFrame(
    [
        {
            "hospital": hospital,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "empty_columns": len(empty_columns(df)),
        }
        for hospital, df in dynamic_tables.items()
    ]
).sort_values("hospital").reset_index(drop=True)

display(dataset_overview)


,hospital,rows,columns,empty_columns
0,asic_UK00,11656,106,42
1,asic_UK02,19297,85,15
2,asic_UK03,21942,105,44
3,asic_UK04,12684,74,10
4,asic_UK06,10808,105,98
5,asic_UK07,20455,106,22
6,asic_UK08,4805,105,46


## Raw Schema Comparison

Compare raw columns across hospitals using the first hospital file as the reference.

In [4]:
reference_hospital = next(iter(dynamic_tables))
reference_columns = list(dynamic_tables[reference_hospital].columns)
reference_set = set(reference_columns)

schema_rows = []
column_sets = [set(df.columns) for df in dynamic_tables.values()]
common_columns = sorted(set.intersection(*column_sets))
all_columns = sorted(set.union(*column_sets))

for hospital, df in dynamic_tables.items():
    columns = list(df.columns)
    column_set = set(columns)
    schema_rows.append(
        {
            "hospital": hospital,
            "same_column_order_as_reference": columns == reference_columns,
            "same_column_set_as_reference": column_set == reference_set,
            "missing_vs_reference": sorted(reference_set - column_set),
            "extra_vs_reference": sorted(column_set - reference_set),
        }
    )

display(pd.DataFrame(schema_rows).sort_values("hospital").reset_index(drop=True))
print(f"Reference hospital: {reference_hospital}")
print(f"Columns common to all hospitals ({len(common_columns)}):")
print(common_columns)
print()
print(f"Columns appearing in at least one hospital ({len(all_columns)}):")
print(all_columns)


,hospital,same_column_order_as_reference,same_column_set_as_reference,missing_vs_reference,extra_vs_reference
0,asic_UK00,True,True,[],[]
1,asic_UK02,False,False,"[AF_spontan, BNP, Bilirubin_ges., CK, DPAP, De...","[Creatinkinase, ECMO_FiO2, Interleukin_6, Koer..."
2,asic_UK03,False,False,[ARDS_Diagnose_App],[]
3,asic_UK04,False,False,"[Clonidin_intravenoes_kontinuierlich, Dexameth...",[Pseudo-ID]
4,asic_UK06,False,False,"[SOFA, Vt]",[Organversagen:_SOFA_Score_ohne_GCS]
5,asic_UK07,False,False,[SOFA],[Organversagen:_SOFA_Score_ohne_GCS]
6,asic_UK08,False,False,"[SOFA, Vt]",[Organversagen:_SOFA_Score_ohne_GCS]


Reference hospital: asic_UK00
Columns common to all hospitals (56):
['24h-Bilanz_(Fluessigkeiten-Einfuhr_vs_-Ausfuhr)', 'AF', 'Albumin', 'Amylase', 'BE_arteriell', 'Bicarbonat_arteriell', 'CK-MB', 'CRP', 'Compliance', 'D-Dimere', 'DAP', 'EVLWI', 'FeO2', 'FiO2', 'GEDVI', 'GOT', 'GPT', 'HF', 'HI_(Bolus)', 'HI_(kontinuierlich)', 'HZV_(Bolus)', 'Haematokrit', 'Haemoglobin', 'Harnstoff', 'Horowitz-Quotient_(ohne_Temp-Korrektur)', 'I:E', 'INR', 'Kreatinin', 'LDH', 'Lagerungstherapie', 'Laktat_arteriell', 'Leukozyten', 'Lipase', 'MAP', 'NT-pro_BNP', 'PCT', 'PEEP', 'P_EI', 'PseudoID', 'SAP', 'SVRI', 'SV_(Bolus)', 'SpO2', 'SzvO2', 'Thrombozyten', 'Time', 'Troponin', 'ZVD', 'Zeit_ab_Aufnahme', 'deltaP', 'etCO2', 'individuelles_Tidalvolumen_pro_kg_idealem_Koerpergewicht', 'pH_arteriell', 'pTT', 'paCO2_(ohne_Temp-Korrektur)', 'paO2_(ohne_Temp-Korrektur)']

Columns appearing in at least one hospital (114):
['24h-Bilanz_(Fluessigkeiten-Einfuhr_vs_-Ausfuhr)', 'AF', 'AF_spontan', 'ARDS_Diagnose_App', 

## Translation Coverage

Apply the shared ASIC translation map and show any raw columns that are still intentionally untranslated.

In [5]:
raw_dynamic_columns = sorted(set().union(*[set(df.columns) for df in dynamic_tables.values()]))
mapped_translation_keys = sorted(dynamic_translation.keys())
unmapped_raw_columns = sorted(set(raw_dynamic_columns) - set(mapped_translation_keys))
extra_translation_keys = sorted(set(mapped_translation_keys) - set(raw_dynamic_columns))

display(
    pd.DataFrame(
        [
            {
                "raw_dynamic_columns": len(raw_dynamic_columns),
                "mapped_translation_keys": len(mapped_translation_keys),
                "unmapped_raw_columns": unmapped_raw_columns,
                "extra_translation_keys": extra_translation_keys,
            }
        ]
    )
)

if unmapped_raw_columns:
    print("Dropped raw dynamic columns (currently not translated):")
    print(unmapped_raw_columns)

translation_df = pd.DataFrame(
    [
        {"raw_column": raw_column, "canonical_name": canonical_name}
        for raw_column, canonical_name in sorted(dynamic_translation.items(), key=lambda item: (item[1], item[0]))
    ]
)
display(translation_df)


,raw_dynamic_columns,mapped_translation_keys,unmapped_raw_columns,extra_translation_keys
0,114,113,[ARDS_Diagnose_App],[]


Dropped raw dynamic columns (currently not translated):
['ARDS_Diagnose_App']


,raw_column,canonical_name
0,Albumin,albumin
1,GPT,alt
2,Amylase,amylase
3,GOT,ast
4,BE_arteriell,base_excess_art
...,...,...
108,Harnstoff,urea
109,Vasopressin_intravenoes_kontinuierlich,vasopressin_iv_cont
110,Vt,vt
111,individuelles_Tidalvolumen_pro_kg_idealem_Koer...,vt_per_kg_ibw


## ECMO vs ECMO_FiO2

Check whether the raw `ECMO` indicator and `ECMO_FiO2` behave like the same variable or not.

In [6]:
ecmo_rows = []
for hospital, df in dynamic_tables.items():
    ecmo = pd.to_numeric(df["ECMO"], errors="coerce") if "ECMO" in df.columns else pd.Series(dtype="float64")
    ecmo_fio2 = pd.to_numeric(df["ECMO_FiO2"], errors="coerce") if "ECMO_FiO2" in df.columns else pd.Series(dtype="float64")

    row = {
        "hospital": hospital,
        "has_ecmo": "ECMO" in df.columns,
        "has_ecmo_fio2": "ECMO_FiO2" in df.columns,
        "ecmo_non_null": int(ecmo.notna().sum()) if "ECMO" in df.columns else 0,
        "ecmo_values": sorted(pd.Series(ecmo.dropna().unique()).astype(float).tolist())[:10] if "ECMO" in df.columns else [],
        "ecmo_fio2_non_null": int(ecmo_fio2.notna().sum()) if "ECMO_FiO2" in df.columns else 0,
        "ecmo_fio2_values": sorted(pd.Series(ecmo_fio2.dropna().unique()).astype(float).tolist())[:10] if "ECMO_FiO2" in df.columns else [],
    }

    if "ECMO" in df.columns and "ECMO_FiO2" in df.columns:
        overlap = ecmo.notna() & ecmo_fio2.notna()
        row["rows_with_both_non_null"] = int(overlap.sum())
        row["rows_ecmo1_and_fio2_non_null"] = int(((ecmo == 1) & ecmo_fio2.notna()).sum())
        row["rows_ecmo0_and_fio2_non_null"] = int(((ecmo == 0) & ecmo_fio2.notna()).sum())
    else:
        row["rows_with_both_non_null"] = 0
        row["rows_ecmo1_and_fio2_non_null"] = 0
        row["rows_ecmo0_and_fio2_non_null"] = 0

    ecmo_rows.append(row)

display(pd.DataFrame(ecmo_rows).sort_values("hospital").reset_index(drop=True))

for hospital, df in dynamic_tables.items():
    if "ECMO" in df.columns and "ECMO_FiO2" in df.columns:
        ecmo = pd.to_numeric(df["ECMO"], errors="coerce")
        ecmo_fio2 = pd.to_numeric(df["ECMO_FiO2"], errors="coerce")
        overlap = ecmo.notna() & ecmo_fio2.notna()
        display(Markdown(f"### {hospital}"))
        if overlap.any():
            display(pd.crosstab(ecmo[overlap], ecmo_fio2[overlap]))
        else:
            print("No overlapping non-null ECMO and ECMO_FiO2 rows.")


,hospital,has_ecmo,has_ecmo_fio2,ecmo_non_null,ecmo_values,ecmo_fio2_non_null,ecmo_fio2_values,rows_with_both_non_null,rows_ecmo1_and_fio2_non_null,rows_ecmo0_and_fio2_non_null
0,asic_UK00,True,False,10096,[0.0],0,[],0,0,0
1,asic_UK02,True,True,6216,"[0.0, 1.0]",617,"[1.0, 21.0, 100.0]",617,617,0
2,asic_UK03,True,False,512,"[0.0, 1.0]",0,[],0,0,0
3,asic_UK04,False,False,0,[],0,[],0,0,0
4,asic_UK06,True,False,0,[],0,[],0,0,0
5,asic_UK07,True,False,3432,"[0.0, 1.0]",0,[],0,0,0
6,asic_UK08,True,False,0,[],0,[],0,0,0


### asic_UK02

ECMO_FiO2,1.0,21.0,100.0
ECMO,,,
1.0,35,2,580


## Theme Groups

Order the translated dynamic columns by broad clinical families.

In [7]:
THEME_GROUPS = {
    "id_time": [
        "hospital_id",
        "stay_id",
        "time",
        "minutes_since_admit",
    ],
    "vitals": [
        "heart_rate",
        "sbp",
        "map",
        "dbp",
        "resp_rate",
        "spont_resp_rate",
        "core_temp",
        "spo2",
        "sao2",
        "scvo2",
        "cvp",
    ],
    "ventilation": [
        "fio2",
        "feo2",
        "peep",
        "delta_p",
        "insp_pressure",
        "compliance",
        "ie_ratio",
        "vt",
        "vt_per_kg_ibw",
        "etco2",
        "pf_ratio",
    ],
    "blood_gas": [
        "pao2",
        "paco2",
        "ph_art",
        "bicarbonate_art",
        "base_excess_art",
        "lactate_art",
    ],
    "hemodynamics": [
        "pap_sys",
        "pap_mean",
        "pap_dias",
        "pcwp",
        "cardiac_output_bolus",
        "cardiac_output_cont",
        "cardiac_index_bolus",
        "cardiac_index_cont",
        "stroke_volume_bolus",
        "stroke_volume_cont",
        "stroke_index_bolus",
        "stroke_index_cont",
        "svri",
        "pvri",
        "gedvi",
        "evlwi",
    ],
    "hematology_coag": [
        "hemoglobin",
        "hematocrit",
        "wbc",
        "platelets",
        "lymph_abs",
        "lymph_pct",
        "inr",
        "ptt",
        "d_dimer",
    ],
    "chemistry_inflammation": [
        "albumin",
        "crp",
        "pct",
        "il6",
        "bilirubin_total",
        "urea",
        "creatinine",
        "bnp",
        "ntprobnp",
        "ast",
        "alt",
        "ldh",
        "amylase",
        "lipase",
        "troponin",
        "ck",
        "ck_mb",
    ],
    "support_therapy": [
        "fluid_balance_24h",
        "position_therapy",
        "ecmo",
        "ecmo_o2",
        "extracorp_blood_flow",
        "extracorp_o2_flow",
        "inhaled_no",
        "inhaled_iloprost",
    ],
    "vasoactive_inotrope": [
        "dobutamine_iv_cont",
        "epinephrine_iv_cont",
        "norepinephrine_iv_cont",
        "vasopressin_iv_cont",
        "milrinone_iv_cont",
        "levosimendan_iv_cont",
        "terlipressin_iv_bolus",
    ],
    "sedation_analgesia": [
        "propofol_iv_cont",
        "midazolam_iv_cont",
        "clonidine_iv_cont",
        "dexmedetomidine_iv_cont",
        "ketanest_iv_cont",
        "isoflurane_inh",
        "sevoflurane_inh",
        "sufentanil_iv_cont",
        "fentanyl_iv_cont",
        "morphine_iv_cont",
        "rocuronium_iv_bolus",
    ],
    "other_meds": [
        "furosemide_iv_cont",
        "hydrocortisone_iv_bolus",
        "prednisolone_iv_bolus",
        "dexamethasone_iv_bolus",
        "fludrocortisone_po_bolus",
    ],
    "scores": [
        "sofa",
    ],
}

canonical_dynamic_columns = sorted(set(dynamic_translation.values()))
assigned_columns = [column for columns in THEME_GROUPS.values() for column in columns]
unassigned_canonical_columns = sorted(set(canonical_dynamic_columns) - set(assigned_columns))

ordered_canonical_columns = []
for theme_columns in THEME_GROUPS.values():
    for column in theme_columns:
        if column == "hospital_id" or column in canonical_dynamic_columns:
            ordered_canonical_columns.append(column)

theme_rows = [{"theme": theme, "columns": columns} for theme, columns in THEME_GROUPS.items()]
display(pd.DataFrame(theme_rows))
print("Unassigned canonical columns:", unassigned_canonical_columns)


,theme,columns
0,id_time,"[hospital_id, stay_id, time, minutes_since_admit]"
1,vitals,"[heart_rate, sbp, map, dbp, resp_rate, spont_r..."
2,ventilation,"[fio2, feo2, peep, delta_p, insp_pressure, com..."
3,blood_gas,"[pao2, paco2, ph_art, bicarbonate_art, base_ex..."
4,hemodynamics,"[pap_sys, pap_mean, pap_dias, pcwp, cardiac_ou..."
5,hematology_coag,"[hemoglobin, hematocrit, wbc, platelets, lymph..."
6,chemistry_inflammation,"[albumin, crp, pct, il6, bilirubin_total, urea..."
7,support_therapy,"[fluid_balance_24h, position_therapy, ecmo, ec..."
8,vasoactive_inotrope,"[dobutamine_iv_cont, epinephrine_iv_cont, nore..."
9,sedation_analgesia,"[propofol_iv_cont, midazolam_iv_cont, clonidin..."


Unassigned canonical columns: []


## Draft Harmonized Dynamic View

Build one harmonized table per hospital using the translation file, keep the `UK02` `I:E` ratio values, and order columns by theme.

In [8]:
harmonized_tables = {}
source_map_rows = []

for hospital, df in dynamic_tables.items():
    harmonized_df, source_map = build_harmonized_dynamic_table(df, hospital, dynamic_translation)
    harmonized_df = harmonized_df[[column for column in ordered_canonical_columns if column in harmonized_df.columns]].copy()
    harmonized_df["stay_id"] = pd.to_numeric(harmonized_df["stay_id"], errors="coerce").astype("Int64")
    harmonized_tables[hospital] = harmonized_df

    for canonical_name in harmonized_df.columns:
        source_map_rows.append(
            {
                "hospital": hospital,
                "canonical_name": canonical_name,
                "raw_source_columns_used": source_map.get(canonical_name, []),
            }
        )

source_map_df = pd.DataFrame(source_map_rows)
combined_harmonized_df = pd.concat(harmonized_tables.values(), ignore_index=True)

display(source_map_df.head(60))
display(combined_harmonized_df.head())


/var/folders/h3/6lrwxtp942d5w3h70w3y31z40000gp/T/ipykernel_4227/1997355555.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_harmonized_df = pd.concat(harmonized_tables.values(), ignore_index=True)


,hospital,canonical_name,raw_source_columns_used
0,asic_UK00,hospital_id,[derived from filename]
1,asic_UK00,stay_id,[PseudoID]
2,asic_UK00,time,[Time]
3,asic_UK00,minutes_since_admit,[Zeit_ab_Aufnahme]
4,asic_UK00,heart_rate,[HF]
5,asic_UK00,sbp,[SAP]
6,asic_UK00,map,[MAP]
7,asic_UK00,dbp,[DAP]
8,asic_UK00,resp_rate,[AF]
9,asic_UK00,spont_resp_rate,[AF_spontan]


,hospital_id,stay_id,time,minutes_since_admit,heart_rate,sbp,map,dbp,resp_rate,spont_resp_rate,...,sufentanil_iv_cont,fentanyl_iv_cont,morphine_iv_cont,rocuronium_iv_bolus,furosemide_iv_cont,hydrocortisone_iv_bolus,prednisolone_iv_bolus,dexamethasone_iv_bolus,fludrocortisone_po_bolus,sofa
0,asic_UK00,372,0 days 00:00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
1,asic_UK00,372,0 days 00:15:00,15.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
2,asic_UK00,372,0 days 00:30:00,30.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
3,asic_UK00,372,0 days 00:45:00,45.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
4,asic_UK00,372,0 days 01:00:00,60.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN


In [9]:
schema_summary = pd.DataFrame(
    [
        {
            "hospital": hospital,
            "rows": df.shape[0],
            "canonical_columns": df.shape[1],
            "empty_columns": empty_columns(df),
        }
        for hospital, df in harmonized_tables.items()
    ]
).sort_values("hospital").reset_index(drop=True)
display(schema_summary)

source_map_by_theme = source_map_df.merge(
    pd.DataFrame(
        [
            {"theme": theme, "canonical_name": canonical_name}
            for theme, columns in THEME_GROUPS.items()
            for canonical_name in columns
        ]
    ),
    how="left",
    on="canonical_name",
)
theme_source_map_display = source_map_by_theme[["theme", "canonical_name", "raw_source_columns_used"]].copy()
theme_source_map_display["raw_source_columns_used"] = theme_source_map_display["raw_source_columns_used"].apply(tuple)
display(theme_source_map_display.drop_duplicates())


,hospital,rows,canonical_columns,empty_columns
0,asic_UK00,11656,106,"[feo2, pap_sys, pap_mean, pap_dias, pcwp, card..."
1,asic_UK02,19297,106,"[spont_resp_rate, sao2, feo2, delta_p, vt, ph_..."
2,asic_UK03,21942,106,"[sbp, map, dbp, feo2, ie_ratio, pap_sys, pap_m..."
3,asic_UK04,12684,106,"[scvo2, feo2, vt, pcwp, cardiac_output_cont, s..."
4,asic_UK06,10808,106,"[heart_rate, sbp, map, dbp, resp_rate, spont_r..."
5,asic_UK07,20455,106,"[feo2, compliance, pcwp, cardiac_output_cont, ..."
6,asic_UK08,4805,106,"[scvo2, cvp, feo2, delta_p, vt, etco2, pcwp, c..."


,theme,canonical_name,raw_source_columns_used
0,id_time,hospital_id,"(derived from filename,)"
1,id_time,stay_id,"(PseudoID,)"
2,id_time,time,"(Time,)"
3,id_time,minutes_since_admit,"(Zeit_ab_Aufnahme,)"
4,vitals,heart_rate,"(HF,)"
...,...,...,...
416,sedation_analgesia,morphine_iv_cont,()
417,sedation_analgesia,rocuronium_iv_bolus,()
419,other_meds,hydrocortisone_iv_bolus,()
422,other_meds,fludrocortisone_po_bolus,()


## Cross-Hospital QC

Surface hospital-column pairs that need a closer look: raw non-numeric values and suspicious cross-hospital distribution mismatches.

In [10]:
non_numeric_issues = find_non_numeric_value_issues(dynamic_tables=dynamic_tables, translation=dynamic_translation)
display(non_numeric_issues)

distribution_summary = numeric_distribution_summary(harmonized_tables, min_non_null=20)
distribution_issues = flag_cross_hospital_distribution_issues(
    distribution_summary,
    min_hospitals=4,
    fence_factor=1.5,
)
display(distribution_issues)


,hospital,raw_column,canonical_name,non_null_count,raw_non_numeric_count,resolved_by_custom_parser_count,unresolved_after_custom_parser_count,non_numeric_examples
0,asic_UK02,I:E,ie_ratio,957,957,957,0,"[1/1.5, 1/2.3, 1/1.4, 1.5/1, 1/1, 1/1.1, 1.1/1..."
1,asic_UK04,Bilirubin_ges.,bilirubin_total,56,5,5,0,[<0.15]
2,asic_UK04,Lymphozyten_absolut,lymph_abs,14,4,4,0,[storniert]
3,asic_UK04,Lymphozyten_prozentual,lymph_pct,14,4,4,0,[storniert]
4,asic_UK04,CRP,crp,148,1,1,0,[<0.1]


,canonical_name,hospital,n,flagged_metrics,min,median,iqr,max,range_width
0,albumin,asic_UK00,57,[max],225.699600,376.166000,120.373120,526.632400,300.932800
1,alt,asic_UK03,212,"[iqr, max, range_width]",6.599000,35.694000,87.885000,6142.976000,6136.377000
2,alt,asic_UK04,60,[min],11.000000,24.500000,39.750000,179.000000,168.000000
3,alt,asic_UK08,25,"[min, iqr]",0.000000,22.000000,18.000000,327.000000,327.000000
4,ast,asic_UK02,181,"[max, range_width]",12.000000,51.590000,75.590000,6180.560000,6168.560000
...,...,...,...,...,...,...,...,...,...
61,sufentanil_iv_cont,asic_UK08,2021,[iqr],0.000000,20.000000,47.470000,74.970000,74.970000
62,troponin,asic_UK02,25,"[min, iqr, max, range_width]",0.020000,0.290000,9.590000,15.080000,15.060000
63,urea,asic_UK08,52,[median],0.000000,22.000000,13.175000,44.100000,44.100000
64,vt_per_kg_ibw,asic_UK00,858,[min],5.822485,6.248122,0.248122,6.496243,0.673758


In [11]:
print(distribution_issues.canonical_name.unique())

['albumin' 'alt' 'ast' 'bilirubin_total' 'ck' 'ck_mb' 'clonidine_iv_cont'
 'compliance' 'core_temp' 'creatinine' 'delta_p' 'ecmo' 'etco2' 'fio2'
 'hematocrit' 'hemoglobin' 'ie_ratio' 'inr' 'insp_pressure' 'lactate_art'
 'ldh' 'lipase' 'map' 'minutes_since_admit' 'norepinephrine_iv_cont' 'pct'
 'peep' 'pf_ratio' 'ph_art' 'platelets' 'propofol_iv_cont' 'ptt'
 'resp_rate' 'sao2' 'sbp' 'spo2' 'spont_resp_rate' 'sufentanil_iv_cont'
 'troponin' 'urea' 'vt_per_kg_ibw']


In [12]:
for col in distribution_issues.canonical_name.unique():
    # Display col distributions across hospitals for a specific column of interest, e.g. 'albumin'
    col_summary = []
    for hospital, df in harmonized_tables.items():
        if col in df.columns:
            col_summary.append(numeric_summary(df[col]))
    col_summary_df = pd.DataFrame(col_summary, index=harmonized_tables.keys()).reset_index().rename(columns={"index": "hospital"})
    print(f"Distribution summary for column '{col}':")
    display(col_summary_df)

Distribution summary for column 'albumin':


,hospital,n,min,q1,median,q3,max,iqr,example_values
0,asic_UK00,57,225.6996,346.07272,376.166,466.44584,526.6324,120.37312,"[496.53912, 315.97944, 511.58576, 466.44584, 4..."
1,asic_UK02,177,0.0,288.89,352.09,424.31,552.21,135.42,"[416.79, 343.06, 281.37, 353.6, 340.05, 296.42..."
2,asic_UK03,123,184.83,292.395,328.755,373.4475,559.035,81.0525,"[490.86, 415.11, 371.175, 193.92, 265.125, 375..."
3,asic_UK04,6,285.886,315.9795,436.3525,522.8705,541.679,206.891,"[511.586, 526.632, 541.679, 300.933, 285.88599..."
4,asic_UK06,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,[]
5,asic_UK07,154,0.78,359.43,390.585,439.14,555.08,79.71,"[410.15, 405.8, 444.94, 394.21, 442.04, 420.3,..."
6,asic_UK08,8,34607.27,35886.24,36713.805,41829.655,50857.64,5943.415,"[44989.45, 35209.14, 36864.27, 40776.39, 50857..."


Distribution summary for column 'alt':


,hospital,n,min,q1,median,q3,max,iqr,example_values
0,asic_UK00,133,7.0,21.0,38.0,61.0,130.0,40.0,"[43.0, 38.0, 51.0, 21.0, 18.0, 17.0, 25.0, 24...."
1,asic_UK02,13,15.6,115.78,952.01,1122.98,1451.11,1007.2,"[323.34, 1220.76, 1451.11, 1122.98, 1024.0, 95..."
2,asic_UK03,212,6.599,20.847,35.694,108.732,6142.976,87.885,"[41.993, 40.793, 34.794000000000004, 47.992, 1..."
3,asic_UK04,60,11.0,16.0,24.5,55.75,179.0,39.75,"[22.0, 14.0, 16.0, 12.0, 91.0, 84.0, 50.0, 36...."
4,asic_UK06,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,[]
5,asic_UK07,228,6.0,24.0,39.0,58.0,328.0,34.0,"[57.0, 309.0, 328.0, 200.0, 161.0, 138.0, 83.0..."
6,asic_UK08,25,0.0,18.0,22.0,36.0,327.0,18.0,"[47.0, 41.0, 36.0, 38.0, 31.0, 39.0, 21.0, 15...."


Distribution summary for column 'ast':


...

,hospital,n,min,q1,median,q3,max,iqr,example_values
0,asic_UK00,32,0.006,0.01575,0.023,0.05925,5.748,0.0435,"[0.014, 0.008, 0.006, 0.022, 0.025, 0.023, 0.0..."
1,asic_UK02,25,0.02,0.04,0.29,9.63,15.08,9.59,"[0.07, 0.11, 0.12, 0.16, 0.18, 1.35, 4.54, 5.5..."
2,asic_UK03,32,0.00773,0.031375,0.06095,0.1985,1.392,0.167125,"[0.421, 0.222, 0.0435, 0.0681, 0.0577, 0.0568,..."
3,asic_UK04,22,0.007,0.01625,0.041,0.47625,1.901,0.46,"[0.011, 0.016, 0.0069999999999999, 0.01, 0.057..."
4,asic_UK06,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,[]
5,asic_UK07,89,0.01,0.17,0.46,1.37,6.2,1.2,"[0.01, 0.06, 0.04, 0.02, 0.71, 5.32, 4.42, 3.6..."
6,asic_UK08,3,0.33,0.375,0.42,0.52,0.62,0.145,"[0.42, 0.33, 0.62]"


Distribution summary for column 'urea':


,hospital,n,min,q1,median,q3,max,iqr,example_values
0,asic_UK00,131,3.006,7.014,10.02,16.1155,42.585,9.1015,"[4.509, 3.841, 5.3439999999999985, 34.73600000..."
1,asic_UK02,177,0.9,6.3,8.5,11.6,33.5,5.3,"[11.1, 11.5, 12.6, 11.3, 12.0, 9.3, 8.4, 9.2, ..."
2,asic_UK03,254,1.3,6.1,9.45,13.55,32.5,7.45,"[11.3, 11.5, 12.0, 16.3, 10.6, 17.4, 11.2, 8.1..."
3,asic_UK04,60,1.837,3.841,5.2605,7.7655,13.694,3.9245,"[2.839, 2.338, 2.6719999999999997, 5.01, 5.343..."
4,asic_UK06,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,[]
5,asic_UK07,237,2.92,5.23,7.58,10.97,27.1,5.74,"[5.91, 5.9, 5.34, 3.71, 5.51, 4.49, 4.66, 6.38..."
6,asic_UK08,52,0.0,16.075,22.0,29.25,44.1,13.175,"[16.0, 30.0, 33.0, 21.0, 12.0, 13.0, 44.0, 34...."


Distribution summary for column 'vt_per_kg_ibw':


,hospital,n,min,q1,median,q3,max,iqr,example_values
0,asic_UK00,858,5.822485,6.248122,6.248122,6.496243,6.496243,0.248122,"[6.248121667474552, 6.36343788274136, 6.165414..."
1,asic_UK02,1842,0.000000,4.550000,6.295000,7.580000,28.940000,3.030000,"[7.42, 4.28, 4.95, 6.52, 6.3, 6.85, 5.87, 5.56..."
2,asic_UK03,7017,0.000000,5.075000,5.800000,6.698000,16.823000,1.623000,"[5.799, 5.2810000000000015, 8.705, 7.997000000..."
3,asic_UK04,3082,0.013000,6.645250,7.638000,8.981500,41.056000,2.336250,"[7.698, 9.138, 9.158, 8.738, 9.806, 9.253, 8.1..."
4,asic_UK06,1909,0.000000,8.643899,4769.172071,8785.602947,16978.252575,8776.959048,"[10063.463281, 10643.699002, 10752.4932, 11060..."
5,asic_UK07,15803,0.000000,5.000000,7.100000,8.520000,33.540000,3.520000,"[5.06, 7.84, 7.83, 7.88, 6.79, 7.45, 9.04, 10...."
6,asic_UK08,569,0.000000,5.800000,6.090000,6.370000,28.430000,0.570000,"[6.97, 3.3, 7.79, 7.55, 4.72, 5.46, 6.24, 1.09..."


**Cross-hospital distrbution mismatches:**
`norepinephrine_iv_cont`, `clonidine_iv_con`, `vt_per_kg_ibw`, `etco2`, `ie_ratio`, `lipase`, `pct`

## Harmonized Value Preview

Show a compact value preview for each harmonized hospital table.

In [13]:
for hospital, df in harmonized_tables.items():
    display(Markdown(f"### {hospital}"))
    display(value_preview(df))


### asic_UK00

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,11656,1,[asic_UK00]
1,stay_id,Int64,11656,10,"[372, 419, 1014, 1204, 1955, 2203, 2353, 2578,..."
2,time,string,11656,2774,"[0 days 00:00:00, 0 days 00:15:00, 0 days 00:3..."
3,minutes_since_admit,float64,11656,2774,"[0.0, 15.0, 30.0, 45.0, 60.0, 75.0, 90.0, 105...."
4,heart_rate,float64,9863,152,"[129.5, 135.0, 134.0, 126.33333333333331, 124...."
...,...,...,...,...,...
101,hydrocortisone_iv_bolus,float64,0,0,[]
102,prednisolone_iv_bolus,float64,0,0,[]
103,dexamethasone_iv_bolus,float64,0,0,[]
104,fludrocortisone_po_bolus,float64,0,0,[]


### asic_UK02

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,19297,1,[asic_UK02]
1,stay_id,Int64,19297,10,"[237, 334, 403, 440, 478, 506, 611, 723, ...]"
2,time,string,19297,5859,"[0 days 00:00:00, 0 days 00:15:00, 0 days 00:3..."
3,minutes_since_admit,int64,19297,5859,"[15, 30, 45, 60, 75, 90, 105, 120, ...]"
4,heart_rate,float64,2030,107,"[82.0, 83.0, 69.0, 77.0, 72.67, 74.0, 64.0, 87..."
...,...,...,...,...,...
101,hydrocortisone_iv_bolus,float64,1,1,[100.0]
102,prednisolone_iv_bolus,float64,0,0,[]
103,dexamethasone_iv_bolus,float64,0,0,[]
104,fludrocortisone_po_bolus,float64,0,0,[]


### asic_UK03

...

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10808,1,[asic_UK06]
1,stay_id,Int64,10808,10,"[237, 334, 403, 440, 478, 506, 578, 604, ...]"
2,time,string,10808,2832,"[0 days 00:00:00, 0 days 00:15:00, 0 days 00:3..."
3,minutes_since_admit,int64,10808,2832,"[0, 15, 30, 45, 60, 75, 90, 105, ...]"
4,heart_rate,float64,0,0,[]
...,...,...,...,...,...
101,hydrocortisone_iv_bolus,float64,0,0,[]
102,prednisolone_iv_bolus,float64,0,0,[]
103,dexamethasone_iv_bolus,float64,0,0,[]
104,fludrocortisone_po_bolus,float64,0,0,[]


### asic_UK07

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,20455,1,[asic_UK07]
1,stay_id,Int64,20455,10,"[372, 419, 1014, 1204, 1221, 1258, 1304, 1934,..."
2,time,string,20455,3432,"[0 days 00:00:00, 0 days 00:15:00, 0 days 00:3..."
3,minutes_since_admit,int64,20455,3432,"[15, 30, 45, 60, 75, 90, 105, 120, ...]"
4,heart_rate,float64,20062,1622,"[66.4, 61.33, 57.53, 59.0, 58.13, 62.87, 59.4,..."
...,...,...,...,...,...
101,hydrocortisone_iv_bolus,float64,7,1,[100.0]
102,prednisolone_iv_bolus,float64,0,0,[]
103,dexamethasone_iv_bolus,float64,26,2,"[8.0, 40.0]"
104,fludrocortisone_po_bolus,float64,0,0,[]


### asic_UK08

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,4805,1,[asic_UK08]
1,stay_id,Int64,4805,10,"[237, 403, 478, 611, 723, 2074, 2142, 2316, ...]"
2,time,string,4805,2370,"[0 days 00:00:00, 0 days 00:15:00, 0 days 00:3..."
3,minutes_since_admit,int64,4805,2370,"[15, 30, 45, 60, 75, 90, 105, 120, ...]"
4,heart_rate,float64,4613,228,"[99.0, 100.0, 95.0, 105.0, 101.0, 107.0, 94.5,..."
...,...,...,...,...,...
101,hydrocortisone_iv_bolus,float64,251,3,"[8.4, 6.2, 4.0]"
102,prednisolone_iv_bolus,float64,0,0,[]
103,dexamethasone_iv_bolus,float64,1,1,[4.0]
104,fludrocortisone_po_bolus,float64,0,0,[]
